# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [4]:
# Install MCP CLI + SDK
!pip install -qU "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 10.6 MB/s eta 0:00:00


In [5]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.12.13
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [1]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server named "Demo"
mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    # Implementation of the add tool
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    # Implementation of the greeting resource
    return f"Hello, {name}!"

if __name__ == "__main__":
    # Start the server loop using STDIO transport
    mcp.run(transport='stdio')

Writing server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [2]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Define how to spawn the server process
server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)

def extract_content(payload):
    """Best-effort to pull text from MCP responses."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. Initialize session
            await session.initialize()

            # 2. List resources
            resources = await session.list_resources()
            print("Resources:", [r.uri for r in resources.resources])

            # 3. List tools
            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])

            # 4. Read greeting://hello
            greeting_resp = await session.read_resource("greeting://hello")
            print("Greeting Result:", extract_content(greeting_resp))

            # 5. Call add tool with a=1, b=7
            add_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("Add Tool Result:", extract_content(add_result))

if __name__ == "__main__":
    asyncio.run(run())

Writing client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [6]:
# Final execution of the client to solve the exercise
!python client.py

[07/08/26 19:36:01] INFO     Processing request of type            ]8;id=324843;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=56202;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
Resources: []
                    INFO     Processing request of type            ]8;id=976044;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=693544;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
Tools: ['add']
                    INFO     Processing request of type            ]8;id=870438;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=266936;file:///usr/local/lib/python3.12/dist-packages/mcp/ser

## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.